In [ ]:
!pip install requests

import os
import requests
import json
import socket

In [ ]:
# Load from environment (set GROQ_API_KEY in .env or shell before running)
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
GROQ_MODEL = "llama-3.1-8b-instant"

if not GROQ_API_KEY:
    print("⚠️  Warning: GROQ_API_KEY not set. Groq queries will fail.")

In [40]:
CLASS_LABELS = [
    "Apple___Apple_scab",
    "Apple___Black_rot",
    "Apple___Cedar_apple_rust",
    "Apple___healthy",
    "Grape___Black_rot",
    "Grape___Esca_(Black_Measles)",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)",
    "Grape___healthy",
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy"
]

In [41]:
import random

def simulate_model_output():
    label = random.choice(CLASS_LABELS)
    confidence = round(random.uniform(0.7, 0.98), 2)

    crop, disease = label.split("___")

    return {
        "crop": crop,
        "disease": disease.replace("_", " "),
        "confidence": confidence
    }

model_output = simulate_model_output()
print(model_output)

{'crop': 'Apple', 'disease': 'Black rot', 'confidence': 0.79}


In [42]:
def normalize_confidence(conf):
    if conf <= 1:
        return int(conf * 100)
    return int(conf)

In [43]:
def is_online():
    try:
        socket.create_connection(("8.8.8.8", 53), timeout=3)
        return True
    except:
        return False

In [44]:
SYSTEM_PROMPT = """
You are an expert agricultural advisor for Indian farmers.

STRICT INSTRUCTIONS:
- Output EXACTLY in the given format
- No extra text
- No markdown
- No explanation
- Generate advice dynamically based on disease

FORMAT:

Crop: {crop}
Disease: {disease}
Confidence: {confidence}%
Severity: <Low/Moderate/High>
Location: <Indian city, state , India>
Time Context: <Best time for action>

── AI Recommendation ───────────────────────

1. <specific actionable step>
2. <specific actionable step>
3. <specific actionable step>
4. <specific actionable step>
5. <specific actionable step>

Est. Recovery: <time range>
Preventive Note: <short preventive advice>
"""

In [45]:
def query_groq(data):
    url = "https://api.groq.com/openai/v1/chat/completions"

    prompt = SYSTEM_PROMPT.format(
        crop=data["crop"],
        disease=data["disease"],
        confidence=normalize_confidence(data["confidence"])
    )

    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "llama-3.1-8b-instant",
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.3
    }

    response = requests.post(url, headers=headers, json=payload)

    try:
        res_json = response.json()
        return res_json["choices"][0]["message"]["content"]
    except:
        print("GROQ ERROR:", response.text)
        return "❌ Groq failed. Check API key."

In [46]:
def query_ollama(data):
    url = "http://localhost:11434/api/generate"

    prompt = SYSTEM_PROMPT.format(
        crop=data["crop"],
        disease=data["disease"],
        confidence=normalize_confidence(data["confidence"])
    )

    payload = {
        "model": "llama3",
        "prompt": prompt,
        "stream": False
    }

    response = requests.post(url, json=payload)
    return response.json()["response"]

In [47]:
def get_recommendation(data):
    print("Using GROQ only")
    return query_groq(data)

In [48]:
predicted_class, confidence_score = model.predict(image)

# ✅ Convert to LLM format
crop, disease = predicted_class.split("___")

model_output = {
    "crop": crop,
    "disease": disease.replace("_", " "),
    "confidence": confidence_score
}

print("INPUT FROM MODEL:", model_output)

# 🔥 Call LLM
result = get_recommendation(model_output)

print("\nOUTPUT:\n")
print(result)

NameError: name 'model' is not defined